In [1]:



import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import torch
import torch.nn as nn
import gymnasium as gym

In [21]:

class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.SiLU(),
            nn.Linear(64,64),
            nn.SiLU(True),
            nn.Linear(64, output_size)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MemBuffer():
    def __init__(self, capacity=10000):
        self.state = []
        self.actions = []
        self.terminated = []
        self.rewards = []
        self.next_state = []
        self.capacity = capacity
        self.len = 0
    def push(self, s, a, t, r, s_next):
        if self.len >= self.capacity:
            self.state.pop(0)
            self.actions.pop(0)
            self.terminated.pop(0)
            self.rewards.pop(0)
            self.next_state.pop(0)
            self.len -= 1
        self.state.append(s)
        self.actions.append(a)
        self.terminated.append(t)
        self.rewards.append(r)
        self.next_state.append(s_next)
        self.len += 1
    def __len__(self):
        return self.len
    def get(self):
        return self.state, self.actions, self.terminated, self.rewards, self.next_state
    def sample(self, batch_size):
        idx = random.sample(range(self.len), batch_size)
        s = torch.stack([self.state[i] for i in idx])
        a = torch.tensor([self.actions[i] for i in idx], dtype=torch.long)
        t = torch.tensor([self.terminated[i] for i in idx], dtype=torch.float32)
        r = torch.tensor([self.rewards[i] for i in idx], dtype=torch.float32)
        s_next = torch.stack([self.next_state[i] for i in idx])
        return s, a, t, r, s_next
    def clear(self):
        self.state = []
        self.actions = []
        self.terminated = []
        self.rewards = []
        self.next_state = []
        self.len = 0


def select_action(model: nn.Module, s: torch.Tensor, noise_eps: float):
    with torch.no_grad():
        q = model(s.unsqueeze(0))
        q = (q - q.mean(dim=-1, keepdim=True)) / (q.std(dim=-1, keepdim=True) + 1e-6)
        q = q + noise_eps * torch.randn_like(q)
        probs = torch.softmax(q, dim=-1)
    return torch.multinomial(probs, 1).item()
# change it
def state_to_tensor(s):
    me, d, ace = s
    # print(me)
    me_t = torch.nn.functional.one_hot(torch.tensor(me).long(), num_classes=40)
    d_t = torch.nn.functional.one_hot(torch.tensor(d).long(), num_classes=11)
    ace = torch.nn.functional.one_hot(torch.tensor(ace).long(), num_classes=2)
    s_t = torch.cat([me_t, d_t, ace]).float()
    return s_t
def learn(buffer: MemBuffer, model: nn.Module, target_model: nn.Module,
          optim: torch.optim, gamma: float = 0.95, batch_size: int = 64):
    if len(buffer) < batch_size:
        return
    s, a, t, r, s_next = buffer.sample(batch_size)
    with torch.no_grad():
        q_next = target_model(s_next).max(dim=1).values
        y = r + gamma * (1.0 - t) * q_next
    q = model(s).gather(1, a.unsqueeze(1)).squeeze(1)
    loss = torch.nn.functional.mse_loss(q, y)
    optim.zero_grad()
    loss.backward()
    optim.step()


def rollout(env, model, input_size, output_size, episodes=500, target_sync=10):

    buffer = MemBuffer()
    # model = DQN(input_size, output_size)
    target_model = DQN(input_size, output_size)
    target_model.load_state_dict(model.state_dict())
    target_model.eval()
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)

    ##
    K = 40
    trailing_Kwr_wins = []

    ##
    for episode in range(episodes):
        s, _ = env.reset()
        s = torch.tensor(s, dtype=torch.float32).flatten()
        done = False

        while not done:
            noise_eps = max(0.05, 0.2 - 0.15 * (episode / episodes))
            s_t = state_to_tensor(s)
            a = select_action(model, s_t, noise_eps)
            s_next, r, term, trunc, _ = env.step(a)
            done = bool(term or trunc)
            s_next = torch.tensor(s_next, dtype=torch.float32).flatten()
            buffer.push(state_to_tensor(s), a, float(term), float(r), state_to_tensor(s_next))
            s = s_next
            learn(buffer, model, target_model, optim)

        if episode % target_sync == 0:
            target_model.load_state_dict(model.state_dict())
        trailing_Kwr_wins.append(r)
        l_trailing = len(trailing_Kwr_wins[-K:])
        print(f" {l_trailing} {sum(trailing_Kwr_wins[-K:])/ min(K,l_trailing):.4f} wr: {(torch.sum((torch.tensor(trailing_Kwr_wins[-K:]) > 0).float()) / min(K,l_trailing)):.3f}")
    return model
def eval_argmax(model, env, n=10000):
    wins = 0
    for _ in range(n):
        s, _ = env.reset()
        done = False
        while not done:
            with torch.no_grad():
                q = model(state_to_tensor(s).unsqueeze(0))
            a = int(q.argmax(dim=1).item())
            s, r, term, trunc, _ = env.step(a)
            done = term or trunc
        wins += (r > 0)
    return wins / n


env = gym.make('Blackjack-v1')
model = DQN(53, 2)
#before
print("argmax win rate:", eval_argmax(model, env))
model = rollout(env, model, 53, 2, 5000)
# after training:
print("argmax win rate:", eval_argmax(model, env))

argmax win rate: 0.3617
 1 1.0000 wr: 1.000
 2 0.0000 wr: 0.500
 3 0.3333 wr: 0.667
 4 0.0000 wr: 0.500
 5 0.2000 wr: 0.600
 6 0.0000 wr: 0.500
 7 0.1429 wr: 0.571
 8 0.0000 wr: 0.500
 9 -0.1111 wr: 0.444
 10 0.0000 wr: 0.500
 11 -0.0909 wr: 0.455
 12 -0.1667 wr: 0.417
 13 -0.0769 wr: 0.462
 14 0.0000 wr: 0.500
 15 -0.0667 wr: 0.467
 16 0.0000 wr: 0.500
 17 -0.0588 wr: 0.471
 18 0.0000 wr: 0.500
 19 -0.0526 wr: 0.474
 20 -0.1000 wr: 0.450
 21 -0.1429 wr: 0.429
 22 -0.1818 wr: 0.409
 23 -0.2174 wr: 0.391
 24 -0.2500 wr: 0.375
 25 -0.2800 wr: 0.360
 26 -0.2308 wr: 0.385
 27 -0.2593 wr: 0.370
 28 -0.2143 wr: 0.393
 29 -0.2414 wr: 0.379
 30 -0.2667 wr: 0.367
 31 -0.2903 wr: 0.355
 32 -0.3125 wr: 0.344
 33 -0.2727 wr: 0.364
 34 -0.2353 wr: 0.382
 35 -0.2571 wr: 0.371
 36 -0.2778 wr: 0.361
 37 -0.2432 wr: 0.378
 38 -0.2632 wr: 0.368
 39 -0.2308 wr: 0.385
 40 -0.2500 wr: 0.375
 40 -0.2750 wr: 0.350
 40 -0.2750 wr: 0.350
 40 -0.3250 wr: 0.325
 40 -0.2750 wr: 0.350
 40 -0.2750 wr: 0.350
 40 -0.

/tmp/ipykernel_715691/3244468806.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  me_t = torch.nn.functional.one_hot(torch.tensor(me).long(), num_classes=40)
/tmp/ipykernel_715691/3244468806.py:73: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  d_t = torch.nn.functional.one_hot(torch.tensor(d).long(), num_classes=11)
/tmp/ipykernel_715691/3244468806.py:74: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ace = torch.nn.functional.one_hot(torch.tensor(ace).long(), num_classes=2)


 40 -0.1750 wr: 0.375
 40 -0.1750 wr: 0.375
 40 -0.1750 wr: 0.375
 40 -0.1750 wr: 0.375
 40 -0.1250 wr: 0.400
 40 -0.0750 wr: 0.425
 40 -0.0750 wr: 0.425
 40 -0.1250 wr: 0.400
 40 -0.1000 wr: 0.425
 40 -0.1000 wr: 0.425
 40 -0.0500 wr: 0.450
 40 0.0000 wr: 0.475
 40 -0.0500 wr: 0.450
 40 -0.1000 wr: 0.425
 40 -0.1000 wr: 0.425
 40 -0.0500 wr: 0.450
 40 0.0000 wr: 0.475
 40 0.0500 wr: 0.500
 40 0.0500 wr: 0.500
 40 0.0500 wr: 0.500
 40 0.0000 wr: 0.475
 40 0.0000 wr: 0.475
 40 -0.0500 wr: 0.450
 40 -0.0250 wr: 0.450
 40 0.0250 wr: 0.475
 40 0.0250 wr: 0.475
 40 0.0750 wr: 0.500
 40 0.0750 wr: 0.500
 40 0.1250 wr: 0.525
 40 0.1250 wr: 0.525
 40 0.0750 wr: 0.500
 40 0.0500 wr: 0.475
 40 0.0250 wr: 0.450
 40 0.0500 wr: 0.475
 40 0.0000 wr: 0.450
 40 0.0000 wr: 0.450
 40 -0.0500 wr: 0.425
 40 0.0000 wr: 0.450
 40 -0.0250 wr: 0.425
 40 0.0250 wr: 0.450
 40 0.0500 wr: 0.475
 40 0.0500 wr: 0.475
 40 0.1000 wr: 0.500
 40 0.1500 wr: 0.525
 40 0.1000 wr: 0.500
 40 0.1000 wr: 0.500
 40 0.1000 wr: 

argmax win rate: 0.42277


In [4]:
# a = [-1,2,3,4,5,7,8]
# k = 4
# z = a[-k:]
# l = (torch.tensor(a) > 0).float()
# l

In [5]:
%%timeit
y = model(torch.randn(53))

58.9 μs ± 2.27 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [6]:
# s, _ = e.reset()
# print(s)
    


In [7]:
# me, dealer, usable_ace = s
# me, dealer, usable_ace